# Bivariate Questions\n\n1. Does age relate to winning?\n2. Does stance relate to winning?\n3. Does number of previous fights relate to winning?\n4. Does time since last fight relate to winning?\n\n**Framing note:** the original EDA (`docs/results.md`) found a large red-corner recording bias - red is credited with the win in nearly 100% of pre-2010 fights, an artifact of how the source data was assembled, not a real effect. Comparing `red vs blue` diffs against `winner_side` would inherit that bias. Instead, every question here reshapes the data to **one row per fighter-per-fight**, with `won` (1/0) as the outcome for that fighter directly - corner is never part of the comparison, so the bias can't leak in. A side effect: since every fight has exactly one winner and one loser, the population win rate is exactly 50% by construction, so any bucket that deviates from that 0.5 line is a real effect worth explaining."

In [ ]:
import sys
sys.path.insert(0, "../src/data")
sys.path.insert(0, "../src/features")

import pandas as pd
import matplotlib.pyplot as plt

from cleaning import clean_fights
from engineering import add_matchup_features, add_fighter_history_features

master = pd.read_csv("../data/master.csv")
fights = clean_fights(master)
fights = add_matchup_features(fights)
fights = add_fighter_history_features(fights)

r_side = fights[["fight_id", "r_fighter_id", "r_age", "r_stance",
                  "r_prior_fights", "r_days_since_last_fight", "winner_id"]].rename(columns={
    "r_fighter_id": "fighter_id", "r_age": "age", "r_stance": "stance",
    "r_prior_fights": "prior_fights", "r_days_since_last_fight": "days_since_last_fight"})
b_side = fights[["fight_id", "b_fighter_id", "b_age", "b_stance",
                  "b_prior_fights", "b_days_since_last_fight", "winner_id"]].rename(columns={
    "b_fighter_id": "fighter_id", "b_age": "age", "b_stance": "stance",
    "b_prior_fights": "prior_fights", "b_days_since_last_fight": "days_since_last_fight"})
individual = pd.concat([r_side, b_side])
individual["won"] = (individual["fighter_id"] == individual["winner_id"]).astype(int)

individual.shape, individual["won"].mean()

## 1. Does Age Relate to Winning?"

In [ ]:
age_bins = [16, 24, 27, 30, 33, 36, 60]
age_labels = ["18-24", "25-27", "28-30", "31-33", "34-36", "37+"]
individual["age_bucket"] = pd.cut(individual["age"], bins=age_bins, labels=age_labels)

age_win_rate = individual.groupby("age_bucket", observed=True)["won"].agg(["mean", "count"])
print(age_win_rate)
print("\ncorrelation (age, won):", individual[["age", "won"]].corr().iloc[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(age_win_rate.index.astype(str), age_win_rate["mean"])
axes[0].axhline(0.5, color="gray", linestyle="--")
for i, count in enumerate(age_win_rate["count"]):
    axes[0].text(i, age_win_rate["mean"].iloc[i] + 0.01, f"n={count}", ha="center")
axes[0].set_ylabel("win rate")
axes[0].set_title("Win Rate by Age Bucket")

axes[1].boxplot([individual.loc[individual["won"] == 0, "age"].dropna(),
                  individual.loc[individual["won"] == 1, "age"].dropna()],
                 tick_labels=["lost", "won"])
axes[1].set_ylabel("age")
axes[1].set_title("Age by Outcome")
plt.tight_layout()
plt.show()

## 2. Does Stance Relate to Winning?\n\nWatch the sample sizes here, not just the bars: `Sideways` (n=6) and `Open Stance` (n=29) are too small to mean anything. `Unknown` (n=626) is the stance-imputation bucket from `clean_fights` for fighters missing that field - its low win rate almost certainly reflects that missing-stance fighters skew toward lower-profile/lower-record fighters, not that \"having no recorded stance\" causes losing. The real comparison is Orthodox vs Southpaw vs Switch."

In [ ]:
stance_win_rate = individual.groupby("stance", observed=True)["won"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(stance_win_rate)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(stance_win_rate.index.astype(str), stance_win_rate["mean"])
ax.axhline(0.5, color="gray", linestyle="--")
for i, count in enumerate(stance_win_rate["count"]):
    ax.text(i, stance_win_rate["mean"].iloc[i] + 0.01, f"n={count}", ha="center")
ax.set_ylabel("win rate")
ax.set_title("Win Rate by Stance")
plt.tight_layout()
plt.show()

## 3. Does Number of Previous Fights Relate to Winning?\n\nThe raw correlation between `prior_fights` and `won` is going to look close to zero - but that's because the relationship isn't linear, not because there's no relationship. Bucketing shows why: debut fighters (0 prior fights) win less often, win rate rises through the early-to-mid career, then falls again at 16+ fights. That late-career dip is likely fighter aging (see question 1) showing up again here, since long-tenured fighters are also older fighters - not a standalone effect of experience itself."

In [ ]:
experience_bins = [-1, 0, 3, 7, 15, 100]
experience_labels = ["0 (debut)", "1-3", "4-7", "8-15", "16+"]
individual["experience_bucket"] = pd.cut(individual["prior_fights"], bins=experience_bins, labels=experience_labels)

experience_win_rate = individual.groupby("experience_bucket", observed=True)["won"].agg(["mean", "count"])
print(experience_win_rate)
print("\ncorrelation (prior_fights, won):", individual[["prior_fights", "won"]].corr().iloc[0, 1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(experience_win_rate.index.astype(str), experience_win_rate["mean"])
ax.axhline(0.5, color="gray", linestyle="--")
for i, count in enumerate(experience_win_rate["count"]):
    ax.text(i, experience_win_rate["mean"].iloc[i] + 0.01, f"n={count}", ha="center")
ax.set_ylabel("win rate")
ax.set_title("Win Rate by Prior Fight Count")
plt.tight_layout()
plt.show()

## 4. Does Time Since Last Fight Relate to Winning?\n\nDebut fighters have no previous fight, so `days_since_last_fight` is `NaN` for them by design (see `src/features/engineering.py`) and they're naturally excluded here rather than dropped as an afterthought.\n\nCaveat worth stating plainly: a falling win rate with longer layoffs could mean \"ring rust,\" but it could just as easily be reverse causation - fighters coming off a loss or an injury are the ones *forced into* long layoffs in the first place, and the same fighters are more likely to lose again. This bivariate view can't distinguish those two explanations; that needs the actual fight-by-fight sequence, not just gap length."

In [ ]:
layoff_bins = [0, 90, 180, 365, 730, 10000]
layoff_labels = ["<=90d", "91-180d", "181-365d", "366-730d", "731d+"]
individual["layoff_bucket"] = pd.cut(individual["days_since_last_fight"], bins=layoff_bins, labels=layoff_labels)

layoff_win_rate = individual.groupby("layoff_bucket", observed=True)["won"].agg(["mean", "count"])
print(layoff_win_rate)
print("\nexcluded as debuts (no prior fight):", individual["days_since_last_fight"].isna().sum())

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(layoff_win_rate.index.astype(str), layoff_win_rate["mean"])
ax.axhline(0.5, color="gray", linestyle="--")
for i, count in enumerate(layoff_win_rate["count"]):
    ax.text(i, layoff_win_rate["mean"].iloc[i] + 0.01, f"n={count}", ha="center")
ax.set_ylabel("win rate")
ax.set_title("Win Rate by Time Since Last Fight")
plt.tight_layout()
plt.show()